# Attention Analysis with LLM Hook Framework

This notebook demonstrates how to analyze attention patterns in transformer models.

## What You'll Learn
- Monitoring multi-head attention
- Analyzing attention entropy and sparsity
- Visualizing attention patterns
- Identifying attention bottlenecks
- KV cache analysis

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from llm_hooks.core import HookManager
from llm_hooks.attention import AttentionMonitor, KVCacheMonitor
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
%matplotlib inline

## Step 1: Create a Transformer Model

We'll create a simple transformer with multi-head attention.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None, return_attention=False):
        batch_size, seq_len, d_model = x.shape
        
        # Linear projections
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Attention weights
        attn_weights = F.softmax(scores, dim=-1)
        
        # Apply attention to values
        context = torch.matmul(attn_weights, V)
        
        # Reshape and project
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.out_linear(context)
        
        if return_attention:
            return output, attn_weights
        return output


class SimpleTransformer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            MultiHeadAttention(d_model, num_heads) 
            for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


# Create model
model = SimpleTransformer(d_model=512, num_heads=8, num_layers=3)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

## Step 2: Set Up Attention Monitoring

We'll use `AttentionMonitor` to track attention patterns.

In [ ]:
# Create hook manager with attention monitor
manager = HookManager(name="attention_analysis")

# Register attention monitor
attention_hook = AttentionMonitor(
    compute_entropy=True,      # Compute attention entropy
    compute_sparsity=True,     # Compute attention sparsity
    track_head_diversity=True, # Track diversity across heads
    max_samples=10,            # Limit samples to save memory
)

manager.register(attention_hook)
manager.apply_to_model(model)
print("✓ Attention monitoring enabled")

## Step 3: Run Inference and Capture Attention

Let's process some sequences through the model.

In [ ]:
# Create sample input (batch_size=2, seq_len=32, d_model=512)
batch_size = 2
seq_len = 32
d_model = 512

x = torch.randn(batch_size, seq_len, d_model)

# Run inference
with torch.no_grad():
    output = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print("\n✓ Inference complete, attention data captured")

## Step 4: Analyze Attention Patterns

Now let's examine the captured attention statistics.

In [ ]:
# Get results
results = manager.get_results()

print(f"Total results: {len(results.results)}\n")

# Analyze each attention layer
for result in results.results:
    if result.hook_type == 'attention':
        print(f"{'='*60}")
        print(f"Layer: {result.layer_name}")
        
        if 'attention_stats' in result.data:
            stats = result.data['attention_stats']
            print(f"\nAttention Statistics:")
            print(f"  Entropy (mean): {stats.get('entropy_mean', 'N/A'):.4f}")
            print(f"  Entropy (std):  {stats.get('entropy_std', 'N/A'):.4f}")
            print(f"  Sparsity:       {stats.get('sparsity', 'N/A'):.2%}")
            
            if 'head_diversity' in stats:
                print(f"  Head Diversity: {stats['head_diversity']:.4f}")
        print()

## Step 5: Visualize Attention Entropy

Let's visualize how attention entropy varies across layers and heads.

In [ ]:
# Extract entropy data
layer_entropies = []
layer_names = []
layer_sparsities = []

for result in results.results:
    if result.hook_type == 'attention' and 'attention_stats' in result.data:
        stats = result.data['attention_stats']
        layer_names.append(result.layer_name.split('.')[-1])
        layer_entropies.append(stats.get('entropy_mean', 0))
        layer_sparsities.append(stats.get('sparsity', 0))

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entropy plot
axes[0].bar(range(len(layer_names)), layer_entropies, color='steelblue')
axes[0].set_xlabel('Attention Layer')
axes[0].set_ylabel('Mean Entropy')
axes[0].set_title('Attention Entropy by Layer')
axes[0].set_xticks(range(len(layer_names)))
axes[0].set_xticklabels(layer_names, rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Sparsity plot
axes[1].bar(range(len(layer_names)), layer_sparsities, color='coral')
axes[1].set_xlabel('Attention Layer')
axes[1].set_ylabel('Sparsity')
axes[1].set_title('Attention Sparsity by Layer')
axes[1].set_xticks(range(len(layer_names)))
axes[1].set_xticklabels(layer_names, rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Visualize Attention Weights (if captured)

If we captured raw attention weights, we can visualize them as heatmaps.

In [ ]:
# Find result with attention weights
for result in results.results:
    if result.hook_type == 'attention' and 'attention_weights' in result.data:
        # Get attention weights (shape: [batch, num_heads, seq_len, seq_len])
        attn_weights = result.data['attention_weights']
        
        # Plot first sample, first head
        if isinstance(attn_weights, torch.Tensor):
            attn_weights = attn_weights.cpu().numpy()
        
        first_head = attn_weights[0, 0]  # First sample, first head
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(first_head, cmap='viridis', square=True, 
                   cbar_kws={'label': 'Attention Weight'})
        plt.xlabel('Key Position')
        plt.ylabel('Query Position')
        plt.title(f'Attention Weights - {result.layer_name} (Head 0)')
        plt.show()
        
        break  # Show only one example

## Step 7: KV Cache Analysis

For autoregressive models, we can monitor KV cache usage.

In [ ]:
# Create new manager with KV cache monitor
with HookManager() as kv_manager:
    kv_hook = KVCacheMonitor(
        track_size=True,
        track_memory=True,
    )
    
    kv_manager.register(kv_hook)
    kv_manager.apply_to_model(model)
    
    # Simulate autoregressive generation
    cache_results = []
    for step in range(10):
        x = torch.randn(1, step + 1, d_model)  # Growing sequence
        with torch.no_grad():
            _ = model(x)
        cache_results.append(kv_manager.get_results())
    
    print(f"✓ Captured KV cache data for {len(cache_results)} generation steps")

## Interpretation Guide

### Attention Entropy
- **High entropy** (~log(seq_len)): Attention is diffuse, model looks at many tokens
- **Low entropy** (~0): Attention is focused on few tokens
- **Pattern**: Early layers often have higher entropy, later layers more focused

### Attention Sparsity
- **High sparsity** (>0.7): Most attention weights are near zero
- **Low sparsity** (<0.3): Attention is distributed
- **Use case**: High sparsity enables KV cache optimization

### Head Diversity
- **High diversity**: Different heads attend to different patterns (good)
- **Low diversity**: Heads are redundant (can potentially prune)

## Optimization Tips

Based on attention analysis, you can:

1. **Identify redundant heads**: Low diversity → candidates for pruning
2. **Optimize KV cache**: High sparsity → use sparse attention
3. **Debug attention**: Check if model attends to expected positions
4. **Layer analysis**: Compare attention patterns across layers

## Next Steps

Try these exercises:

1. Apply to a real transformer model (e.g., from Hugging Face)
2. Compare attention patterns with different inputs
3. Track how attention changes during training
4. Use attention analysis to guide model pruning

**Next notebook**: `03_gradient_debugging.ipynb` - Debug training with gradient hooks